# CSCE 636 Project 1 — DNN for Estimating m-Height of Analog Code C

This notebook implements a complete **Mixture-of-Experts (MoE)** pipeline to predict the **m-height** of an analog code $C$ given its parameters $(n, k, m, P)$. The approach includes:

1. **LP-based data augmentation** to enlarge the training set
2. **77-dimensional feature engineering** from the parity-check matrix $P$
3. **9 specialized ResidualDNN experts** — one per $(k, m)$ group
4. **Retrained experts** with augmented data + cross-group transfer
5. **Diverse ensemble** with trimmed/weighted evaluation per group
6. **Model saving** and a standalone **inference cell** for grading

All training is done in $\log_2$-space. The cost metric is $\delta(y, \hat{y}) = (\log_2 y - \log_2 \hat{y})^2$.

## Cell 1: Install Dependencies

Install all required Python packages. This cell should be run once at the beginning to ensure all modules are available.

In [ ]:
!pip install torch numpy scipy scikit-learn

## Cell 2: Import Libraries and Configuration

Import all necessary libraries:
- **PyTorch** for building and training the DNN
- **NumPy** for numerical operations and feature engineering
- **SciPy** (`linprog`) for LP-based m-height computation used in data augmentation
- **scikit-learn** (`StandardScaler`) for per-group feature normalization
- Standard libraries (`pickle`, `os`, `time`, etc.) for I/O and utilities

We also define all hyperparameters and configuration constants here for easy tuning.

In [ ]:
import os
import sys
import time
import pickle
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.optimize import linprog
from itertools import combinations
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# Configuration — All hyperparameters in one place
# =============================================================================
SEED = 42
BASE_EPOCHS = 1000
WEAK_EPOCHS = 1200
ENSEMBLE_EPOCHS = 1200
BASE_RUNS = 15
HARD_RUNS = 15
RETRAIN_RUNS = 15
ENSEMBLE_SIZE = 15
LP_PER_GROUP = 10000
LP_WEAK = 15000
BATCH_SIZE = 512
COST_THRESHOLD = 0.0  # 0 = treat ALL groups for retrain + ensemble
NUM_WORKERS = 4
NO_AUGMENT = False
FEATURE_DIM = 77

# Paths — adjust these to match your environment
DATA_DIR = '.'       # directory containing data files
OUTPUT_DIR = '.'     # directory for saving model + predictions

np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device selection
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
    print(f'[GPU] {torch.cuda.get_device_name(0)}')
    print(f'[GPU] Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

## Cell 3: Dataset Class

A simple PyTorch `Dataset` wrapper that converts NumPy feature arrays and target arrays into tensors for efficient batch loading during training.

In [ ]:
class MHeightDataset(Dataset):
    """Simple dataset wrapper for features + targets."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Cell 4: DNN Architecture — ResidualDNN Expert

The core architecture is a **Residual DNN** (`ExpertDNN`) consisting of:

1. **Input projection**: Linear → BatchNorm → ReLU → Dropout to map 77-dim features into the hidden dimension
2. **Residual blocks**: Each block has two Linear+BatchNorm layers with a skip connection and ReLU activation. This allows deeper networks without vanishing gradients.
3. **Output head**: Linear → ReLU → Linear producing a single scalar (predicted $\log_2(\text{m-height})$)

Different $(k,m)$ groups use different configurations (hidden_dim, num_blocks, dropout) depending on difficulty.

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block: two linear layers with BatchNorm + skip connection."""
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.block(x) + x)

class ExpertDNN(nn.Module):
    """ResidualDNN expert: input_proj → N residual blocks → output head."""
    def __init__(self, input_dim, hidden_dim=256, num_blocks=3, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)]
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_blocks(x)
        return self.output_head(x)

## Cell 5: LP-Based m-Height Computation and Data Augmentation

**`compute_m_height_lp(n, k, m, P)`**: Computes the exact m-height of an analog code $C$ with generator matrix $G = [I_k | P]$ using the LP formulation from the project description. For each choice of $m$ columns $S$ from $G$, it solves a linear program to find the maximum absolute value of a coefficient.

**`generate_lp_samples(n, k, m, num_samples)`**: Generates new training samples by:
1. Randomly sampling parity-check matrices $P$ from multiple value ranges (1, 2, 5, ..., 200)
2. Computing the exact m-height via LP for each
3. Filtering out invalid results

This is the data augmentation strategy recommended in the project description to enlarge the training set.

In [ ]:
def compute_m_height_lp(n, k, m, P):
    """Compute exact m-height via LP for code C with G=[I_k|P]."""
    G = np.hstack([np.eye(k), P])
    max_z = 1.0
    for S in combinations(range(n), m):
        S_set = set(S)
        S_bar = [t for t in range(n) if t not in S_set]
        for j in S:
            c = -G[:, j]
            A_ub, b_ub = [], []
            for t in S_bar:
                A_ub.append(G[:, t]);  b_ub.append(1.0)
                A_ub.append(-G[:, t]); b_ub.append(1.0)
            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), method='highs')
            if result.success:
                z = -result.fun
                if z > max_z:
                    max_z = z
    return max_z

def generate_lp_samples(n, k, m, num_samples, p_ranges=None):
    """Generate LP-augmented samples across multiple P-value ranges."""
    if p_ranges is None:
        p_ranges = [1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0]
    new_X, new_y = [], []
    per_range = max(1, num_samples // len(p_ranges))
    for p_range in p_ranges:
        for _ in range(per_range):
            P = np.random.uniform(-p_range, p_range, size=(k, n - k))
            mh = compute_m_height_lp(n, k, m, P)
            if np.isfinite(mh) and mh >= 1.0:
                new_X.append([n, k, m, P])
                new_y.append(mh)
    return new_X, new_y

## Cell 6: Feature Engineering (77-dimensional)

Each sample $(n, k, m, P)$ is transformed into a **77-dimensional feature vector** that captures various structural properties of the parity-check matrix $P$ and the generator matrix $G = [I_k | P]$:

| # | Feature | Dims | Description |
|---|---------|------|-------------|
| 1 | `n, k, m` | 3 | Code parameters |
| 2 | Padded $P$ | 30 | Flattened $6 \times 5$ zero-padded matrix |
| 3 | Row norms | 6 | $\|P_{i,:}\|_2$ for each row |
| 4 | Col norms | 5 | $\|P_{:,j}\|_2$ for each column |
| 5 | SVD | 5 | Top-5 singular values of $P$ |
| 6 | Stats | 5 | Frobenius norm, max/mean/std/min of $|P|$ |
| 7 | Condition | 1 | $\log(1 + \text{cond}(P))$ |
| 8 | Ratio | 1 | Mean row norm / mean col norm |
| 9 | $G$ col norms | 13 | Norms of $G$ columns + max/min/mean/std |
| 10 | Rank | 2 | Matrix rank + effective rank |
| 11 | SV ratios | 3 | $\sigma_1/\sigma_{\min}$, sum, energy ratio |
| 12 | $P^T P$ | 2 | Trace and Frobenius norm of $P^T P$ |
| 13 | Determinant | 1 | $\log(1 + |\det(P_{\text{square}})|)$ |

In [ ]:
def extract_features(sample):
    """Extract 77-dimensional feature vector from a sample [n, k, m, P]."""
    n, k, m, P = sample
    max_k, max_nk, max_sv = 6, 5, 5

    # 1) Padded P (30)
    P_padded = np.zeros((max_k, max_nk))
    P_padded[:k, :n-k] = P
    P_flat = P_padded.flatten()

    # 2) Row norms (6)
    row_norms = np.zeros(max_k)
    for i in range(k):
        row_norms[i] = np.linalg.norm(P[i, :])

    # 3) Col norms (5)
    col_norms = np.zeros(max_nk)
    for j in range(n - k):
        col_norms[j] = np.linalg.norm(P[:, j])

    # 4) SVD (5)
    sv = np.zeros(max_sv)
    try:
        svs = np.linalg.svd(P, compute_uv=False)
        sv[:len(svs)] = np.sort(svs)[::-1]
    except:
        pass

    # 5) Stats (5)
    frob_norm = np.linalg.norm(P, 'fro')
    max_abs = np.max(np.abs(P))
    mean_abs = np.mean(np.abs(P))
    std_abs = np.std(np.abs(P))
    min_abs = np.min(np.abs(P))

    # 6) Cond (1)
    try:
        cond = np.log1p(np.linalg.cond(P))
    except:
        cond = 0.0
    if not np.isfinite(cond):
        cond = 50.0

    # 7) Ratio (1)
    mean_row = np.mean(row_norms[:k]) if k > 0 else 1.0
    mean_col = np.mean(col_norms[:n-k]) if (n - k) > 0 else 1.0
    ratio = mean_row / (mean_col + 1e-8)

    # 8) G=[I|P] col norms (9+4=13)
    G = np.hstack([np.eye(k), P])
    g_col_norms = np.zeros(9)
    for j in range(n):
        g_col_norms[j] = np.linalg.norm(G[:, j])
    g_col_max = np.max(g_col_norms)
    g_col_min = np.min(g_col_norms[:n])
    g_col_mean = np.mean(g_col_norms[:n])
    g_col_std = np.std(g_col_norms[:n])

    # 9) Rank (2)
    try:
        rank = np.linalg.matrix_rank(P)
    except:
        rank = min(k, n - k)
    effective_rank = np.sum(sv > 1e-6)

    # 10) SV ratios (3)
    sv_ratio = sv[0] / (sv[min(k, n-k)-1] + 1e-10) if sv[min(k, n-k)-1] > 1e-10 else 100.0
    sv_sum = np.sum(sv)
    sv_energy_ratio = sv[0] / (sv_sum + 1e-10)

    # 11) PtP features (2)
    try:
        PtP = P.T @ P
        ptP_trace = np.trace(PtP)
        ptP_frob = np.linalg.norm(PtP, 'fro')
    except:
        ptP_trace = frob_norm ** 2
        ptP_frob = frob_norm ** 2

    # 12) Det (1)
    try:
        sq = min(k, n - k)
        det_val = np.log1p(abs(np.linalg.det(P[:sq, :sq])))
    except:
        det_val = 0.0
    if not np.isfinite(det_val):
        det_val = 0.0

    return np.concatenate([
        [n, k, m], P_flat, row_norms, col_norms, sv,
        [frob_norm, max_abs, mean_abs, std_abs, min_abs],
        [cond, ratio],
        g_col_norms, [g_col_max, g_col_min, g_col_mean, g_col_std],
        [rank, effective_rank],
        [sv_ratio, sv_sum, sv_energy_ratio],
        [ptP_trace, ptP_frob],
        [det_val],
    ])

## Cell 7: Loss Functions and Training Helpers

We define several loss functions and training utilities:

- **`log_cosh_loss`**: Smooth approximation to MAE, robust to outliers — $\mathcal{L} = \mathbb{E}[\log(\cosh(\hat{y} - y))]$
- **`combined_loss`**: Weighted combination of 50% Huber (SmoothL1, β=0.5) + 30% MSE + 20% log-cosh. This balances accuracy (MSE), robustness (Huber), and smoothness (log-cosh).
- **`feature_noise`**: Adds Gaussian noise to input features for regularization
- **`train_single_run`**: Core training loop with:
  - AdamW optimizer with weight decay
  - `CosineAnnealingWarmRestarts` scheduler ($T_0=60$, $T_{\text{mult}}=2$)
  - Gradient clipping (`max_norm=1.0`)
  - Early stopping with patience
  - **Mixup augmentation** (α=0.2): randomly interpolates between training pairs for regularization
  - Feature noise injection
- **`batch_predict`**: Efficient batched inference utility

In [ ]:
def log_cosh_loss(pred, target):
    """Log-cosh loss: smooth approximation to MAE."""
    diff = pred - target
    return torch.mean(torch.log(torch.cosh(diff + 1e-12)))

def feature_noise(x, std=0.05):
    """Add Gaussian noise to features for regularization."""
    return x + torch.randn_like(x) * std

def combined_loss(pred, target, beta=0.5):
    """Combined loss: 50% Huber + 30% MSE + 20% log-cosh."""
    return (0.5 * nn.functional.smooth_l1_loss(pred, target, beta=beta)
            + 0.3 * nn.functional.mse_loss(pred, target)
            + 0.2 * log_cosh_loss(pred, target))


def train_single_run(model, train_loader, val_loader, train_size, val_size,
                     device, lr, weight_decay, max_epochs, patience,
                     noise_std=0.0, use_combined_loss=False, seed=42,
                     use_mixup=False, mixup_alpha=0.2):
    """Train a model for one run, return best val loss + state dict."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    if use_combined_loss:
        criterion_fn = lambda p, t: combined_loss(p, t, beta=0.5)
    else:
        criterion = nn.SmoothL1Loss(beta=0.5)
        criterion_fn = lambda p, t: criterion(p, t)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=60, T_mult=2, eta_min=1e-7
    )

    best_val = float('inf')
    pat_cnt = 0
    best_state = None
    t_hist, v_hist = [], []

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            if noise_std > 0:
                bx = feature_noise(bx, std=noise_std)
            # Mixup augmentation: interpolate between random pairs
            if use_mixup and mixup_alpha > 0:
                lam = np.random.beta(mixup_alpha, mixup_alpha)
                idx = torch.randperm(bx.size(0), device=bx.device)
                bx = lam * bx + (1 - lam) * bx[idx]
                by = lam * by + (1 - lam) * by[idx]
            pred = model(bx)
            loss = criterion_fn(pred, by)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * bx.size(0)
        train_loss /= train_size
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                val_loss += nn.functional.mse_loss(model(bx), by).item() * bx.size(0)
        val_loss /= val_size

        t_hist.append(train_loss)
        v_hist.append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            pat_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            pat_cnt += 1

        if (epoch + 1) % 200 == 0 or epoch == 0:
            print(f"      Ep [{epoch+1:4d}/{max_epochs}] Train: {train_loss:.6f} "
                  f"Val: {val_loss:.6f} Best: {best_val:.6f}")

        if pat_cnt >= patience:
            print(f"      Early stop ep {epoch+1}, best: {best_val:.6f}")
            break

    return best_val, best_state, t_hist, v_hist


def batch_predict(model, tensor, batch_size=2048):
    """Efficient batched inference."""
    preds = []
    model.eval()
    with torch.no_grad():
        for bi in range(0, len(tensor), batch_size):
            preds.append(model(tensor[bi:bi+batch_size]).cpu().numpy().flatten())
    return np.concatenate(preds)

## Cell 8: STEP 1 — Load Training Data

Load the provided training data from the two pickle files:
- **`CSCE-636-Project-1-Train-n_k_m_P`**: List of `[n, k, m, P]` samples where $P$ is a $k \times (n-k)$ numpy array
- **`CSCE-636-Project-1-Train-mHeights`**: Corresponding list of m-height values

The dataset has 96,524 samples across 9 $(k,m)$ groups: $k \in \{4,5,6\}$, $m \in \{2,\ldots,n-k\}$ with $n=9$.

In [ ]:
# =========================================================================
# STEP 1: Load Data
# =========================================================================
print('=' * 70)
print('STEP 1: Load Data')
print('=' * 70)

x_path = os.path.join(DATA_DIR, 'CSCE-636-Project-1-Train-n_k_m_P')
y_path = os.path.join(DATA_DIR, 'CSCE-636-Project-1-Train-mHeights')

with open(x_path, 'rb') as f:
    X_raw = pickle.load(f)
with open(y_path, 'rb') as f:
    y_raw = pickle.load(f)

print(f'Loaded {len(X_raw)} samples, y range [{min(y_raw):.4f}, {max(y_raw):.4f}]')
param_counts = Counter([(s[0], s[1], s[2]) for s in X_raw])
for params, count in sorted(param_counts.items()):
    print(f'  n={params[0]}, k={params[1]}, m={params[2]}: {count}')

## Cell 9: STEP 2 — Initial LP Augmentation

For each $(n, k, m)$ group, if the group has fewer than `target_per_group` (20,000) samples, we generate additional samples using the LP-based algorithm. This creates random $P$ matrices from multiple value ranges and computes the exact m-height via linear programming.

This is the data augmentation strategy recommended in the project description: *"you can implement the LP-based algorithm described in the project file to create more samples. Having a larger dataset is usually helpful."*

In [ ]:
# =========================================================================
# STEP 2: Initial LP Augmentation
# =========================================================================
print('\n' + '=' * 70)
print('STEP 2: LP Augmentation (initial)')
print('=' * 70)

X_combined = list(X_raw)
y_combined = list(y_raw)

if not NO_AUGMENT:
    target_per_group = 20000
    for (n_val, k_val, m_val), count in sorted(param_counts.items()):
        deficit = max(0, target_per_group - count)
        num_gen = min(deficit, LP_PER_GROUP)
        if num_gen > 0:
            print(f'  (n={n_val},k={k_val},m={m_val}): +{num_gen} LP samples...', end=' ', flush=True)
            t0 = time.time()
            new_x, new_y = generate_lp_samples(n_val, k_val, m_val, num_gen,
                                                p_ranges=[1, 2, 5, 10, 20, 50, 100])
            X_combined.extend(new_x)
            y_combined.extend(new_y)
            print(f'{len(new_y)} valid ({time.time()-t0:.1f}s)')
        else:
            print(f'  (n={n_val},k={k_val},m={m_val}): {count} — skip')

print(f'Total after augmentation: {len(X_combined)}')

## Cell 10: STEP 3 — Feature Engineering

Apply the 77-dimensional `extract_features` function to every sample (original + augmented). All training targets are converted to $\log_2$-space since the cost metric is $(\log_2 y - \log_2 \hat{y})^2$.

Any NaN or infinite values are clipped to safe ranges.

In [ ]:
# =========================================================================
# STEP 3: Feature Engineering
# =========================================================================
print('\n' + '=' * 70)
print('STEP 3: Feature Engineering (77-dim)')
print('=' * 70)

X_array = np.array([extract_features(s) for s in X_combined], dtype=np.float32)
y_log2 = np.log2(np.array(y_combined, dtype=np.float32)).reshape(-1, 1)
X_array = np.nan_to_num(X_array, nan=0.0, posinf=100.0, neginf=-100.0)

print(f'Features: {X_array.shape}, Targets: {y_log2.shape}')
print(f'log2(y) range: [{y_log2.min():.4f}, {y_log2.max():.4f}]')

## Cell 11: STEP 4 — Per-Group Dataset Preparation

Split the data into 9 separate groups by $(k, m)$. For each group:
1. Apply `StandardScaler` normalization (zero mean, unit variance)
2. Split 90% train / 10% validation (deterministic seed for reproducibility)
3. Create PyTorch `DataLoader`s with batching, shuffling, and pinned memory

In [ ]:
# =========================================================================
# STEP 4: Per-group Dataset Preparation
# =========================================================================
print('\n' + '=' * 70)
print('STEP 4: Per-group Dataset Preparation')
print('=' * 70)

group_keys = sorted(set((s[1], s[2]) for s in X_combined))
print(f'Groups: {group_keys}')

group_data = {}
for (k_val, m_val) in group_keys:
    indices = [i for i, s in enumerate(X_combined) if s[1] == k_val and s[2] == m_val]
    X_grp = X_array[indices]
    y_grp = y_log2[indices]
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_grp)
    np.random.seed(42)
    perm = np.random.permutation(len(X_sc))
    split = int(0.9 * len(perm))
    train_idx, val_idx = perm[:split], perm[split:]
    train_ds = MHeightDataset(X_sc[train_idx], y_grp[train_idx])
    val_ds = MHeightDataset(X_sc[val_idx], y_grp[val_idx])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    group_data[(k_val, m_val)] = {
        'scaler': scaler, 'train_loader': train_loader, 'val_loader': val_loader,
        'train_size': len(train_idx), 'val_size': len(val_idx),
        'feature_dim': X_sc.shape[1],
    }
    print(f'  (k={k_val},m={m_val}): {len(indices)} total, '
          f'{len(train_idx)} train, {len(val_idx)} val')

## Cell 12: STEP 5 — Train 9 Base Experts

Train one specialized `ExpertDNN` per $(k,m)$ group using scaled-up architectures:

| Tier | Groups | Hidden | Blocks | Dropout | LR | Patience | Runs |
|------|--------|--------|--------|---------|----|----------|------|
| Easy | (4,3), (5,2), (6,2) | 768 | 6 | 0.12 | 5e-4 | 60 | 15 |
| Medium | (4,2), (4,4), (5,3) | 1024 | 8 | 0.10 | 3e-4 | 70 | 15 |
| Hard | (4,5), (5,4), (6,3) | 1024 | 8 | 0.10 | 3e-4 | 80 | 15 |

Each expert is trained with **best-of-N** strategy (keep the run with lowest validation loss). All base experts use **Mixup augmentation** (α=0.2) and **feature noise** (σ=0.02) for regularization.

In [ ]:
# =========================================================================
# STEP 5: Train Base Experts (SCALED UP)
# =========================================================================
print('\n' + '=' * 70)
print('STEP 5: Train 9 Base Experts')
print('=' * 70)

GROUP_HPARAMS = {
    # Easy groups
    (4, 3): (768,  6, 0.12, 5e-4, BASE_EPOCHS, 60, BASE_RUNS),
    (5, 2): (768,  6, 0.12, 5e-4, BASE_EPOCHS, 60, BASE_RUNS),
    (6, 2): (768,  6, 0.12, 5e-4, BASE_EPOCHS, 60, BASE_RUNS),
    # Medium groups
    (4, 2): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 70, HARD_RUNS),
    (4, 4): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 70, HARD_RUNS),
    (5, 3): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 70, HARD_RUNS),
    # Hard groups
    (4, 5): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 80, HARD_RUNS),
    (5, 4): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 80, HARD_RUNS),
    (6, 3): (1024, 8, 0.10, 3e-4, BASE_EPOCHS, 80, HARD_RUNS),
}

experts = {}
expert_hparams = {}
t_total = time.time()

for gk in group_keys:
    hparams = GROUP_HPARAMS.get(gk, (512, 5, 0.15, 8e-4, BASE_EPOCHS, 40, BASE_RUNS))
    hidden_dim, num_blocks, dropout, lr, max_epochs, patience, num_runs = hparams
    expert_hparams[gk] = (hidden_dim, num_blocks, dropout)
    info = group_data[gk]

    print(f"\n{'='*60}")
    print(f'Expert (k={gk[0]},m={gk[1]}): h={hidden_dim}, b={num_blocks}, '
          f'd={dropout}, runs={num_runs}')
    print(f'  Data: {info["train_size"]} train, {info["val_size"]} val')
    print(f"{'='*60}")

    global_best_val = float('inf')
    global_best_state = None

    for run in range(num_runs):
        print(f'\n    --- Run {run+1}/{num_runs} ---')
        seed = SEED + run * 777 + gk[0] * 100 + gk[1] * 10
        model = ExpertDNN(info['feature_dim'], hidden_dim, num_blocks, dropout).to(device)
        val_loss, state, _, _ = train_single_run(
            model, info['train_loader'], info['val_loader'],
            info['train_size'], info['val_size'], device,
            lr=lr, weight_decay=1e-4, max_epochs=max_epochs,
            patience=patience, noise_std=0.02,
            use_combined_loss=False, seed=seed,
            use_mixup=True, mixup_alpha=0.2,
        )
        print(f'    Run {run+1} best val: {val_loss:.6f}')
        if val_loss < global_best_val:
            global_best_val = val_loss
            global_best_state = state

    model = ExpertDNN(info['feature_dim'], hidden_dim, num_blocks, dropout).to(device)
    model.load_state_dict(global_best_state)
    model.eval()
    experts[gk] = model
    print(f'  ✓ (k={gk[0]},m={gk[1]}): best_val = {global_best_val:.6f}')

print(f'\n[Base experts] Total time: {time.time()-t_total:.1f}s')

## Cell 13: STEP 6 — Evaluate Base Experts & Identify Weak Groups

Evaluate each base expert on its validation set using the cost metric $(\log_2 y - \log_2 \hat{y})^2$. Groups with cost above `COST_THRESHOLD` (set to 0.0, meaning ALL groups) are marked as "weak" and will receive additional treatment in subsequent steps.

This outputs the **base training performance** for each group.

In [ ]:
# =========================================================================
# STEP 6: Identify Weak Groups + Base Evaluation
# =========================================================================
print('\n' + '=' * 70)
print('STEP 6: Evaluate Base Experts → Identify Weak Groups')
print('=' * 70)

group_costs = {}
total_cost_sum, total_count = 0.0, 0
for gk in group_keys:
    model = experts[gk]; model.eval()
    preds_l, trues_l = [], []
    with torch.no_grad():
        for bx, by in group_data[gk]['val_loader']:
            bx = bx.to(device)
            preds_l.append(model(bx).cpu().numpy())
            trues_l.append(by.numpy())
    pa = np.concatenate(preds_l).flatten()
    ta = np.concatenate(trues_l).flatten()
    cost = ((ta - pa) ** 2).mean()
    group_costs[gk] = cost
    total_cost_sum += ((ta - pa) ** 2).sum()
    total_count += len(ta)
    marker = '★ WEAK' if cost > COST_THRESHOLD else '✓ ok'
    print(f'  (k={gk[0]},m={gk[1]}): cost={cost:.4f} {marker}')

overall_base = total_cost_sum / total_count
print(f'\nOverall base cost: {overall_base:.6f}')

weak_groups = [gk for gk in group_keys if group_costs[gk] > COST_THRESHOLD]
print(f'Weak groups: {weak_groups}')

## Cell 14: STEP 7 — Extra LP Augmentation for Weak Groups

For each weak group (with `COST_THRESHOLD=0.0`, this is **all 9 groups**), generate an additional `LP_WEAK` (15,000) LP-augmented samples with P values drawn from a wide range of scales $[1, 2, 5, 10, 20, 50, 100, 200]$.

This provides a much richer training set for the retrain and ensemble phases. Note: LP computation for groups with large $\binom{n}{m}$ (like $(k=4, m=5)$) can be slow.

In [ ]:
# =========================================================================
# STEP 7: Massive LP Augmentation for Weak Groups
# =========================================================================
if weak_groups:
    print('\n' + '=' * 70)
    print('STEP 7: Extra LP Augmentation for Weak Groups')
    print('=' * 70)

    weak_extra_X, weak_extra_y = {}, {}
    for gk in weak_groups:
        k_val, m_val = gk
        print(f'\n  Generating {LP_WEAK} LP samples for (k={k_val},m={m_val})...',
              flush=True)
        t0 = time.time()
        new_x, new_y = generate_lp_samples(
            9, k_val, m_val, LP_WEAK,
            p_ranges=[1, 2, 5, 10, 20, 50, 100, 200]
        )
        weak_extra_X[gk] = new_x
        weak_extra_y[gk] = new_y
        print(f'    Generated {len(new_y)} samples in {time.time()-t0:.1f}s')
else:
    print('\n✅ No weak groups — skipping extra LP augmentation.')

## Cell 15: STEP 8 — Build Augmented Datasets with Cross-Group Transfer

For each weak group, build an enriched training set by combining:
1. **Original data** from the training set
2. **LP-augmented samples** generated in Step 7
3. **Cross-group transfer**: borrow samples from neighboring $(k,m)$ groups (e.g., $(4,5)$ borrows from $(4,4)$ and $(4,3)$)

Target values are clipped at the 2nd/98th percentile to reduce outlier effects. The validation set uses **only original unclipped data** for fair evaluation.

In [ ]:
# =========================================================================
# STEP 8: Build Augmented Datasets + Cross-Group Transfer
# =========================================================================
if weak_groups:
    print('\n' + '=' * 70)
    print('STEP 8: Build Augmented Datasets with Cross-Group Transfer')
    print('=' * 70)

    cross_group_map = {
        (4, 2): [(4, 3)],
        (4, 3): [(4, 2), (4, 4)],
        (4, 4): [(4, 3), (4, 5)],
        (4, 5): [(4, 4), (4, 3)],
        (5, 2): [(5, 3)],
        (5, 3): [(5, 2), (5, 4)],
        (5, 4): [(5, 3), (5, 2)],
        (6, 2): [(6, 3)],
        (6, 3): [(6, 2)],
    }

    weak_group_data = {}
    for gk in weak_groups:
        k_val, m_val = gk

        # Original data
        orig_indices = [i for i, s in enumerate(X_combined) if s[1] == k_val and s[2] == m_val]
        X_orig = X_array[orig_indices]
        y_orig = y_log2[orig_indices]

        # LP augmented
        extra_feats = np.array([extract_features(s) for s in weak_extra_X[gk]], dtype=np.float32)
        extra_y = np.log2(np.array(weak_extra_y[gk], dtype=np.float32)).reshape(-1, 1)

        # Cross-group transfer
        cross_X_list, cross_y_list = [], []
        for neighbor_gk in cross_group_map.get(gk, []):
            nk, nm = neighbor_gk
            nidx = [i for i, s in enumerate(X_combined) if s[1] == nk and s[2] == nm]
            if nidx:
                np.random.seed(42)
                n_borrow = max(500, len(nidx) // 10)
                chosen = np.random.choice(nidx, size=min(n_borrow, len(nidx)), replace=False)
                cross_X_list.append(X_array[chosen])
                cross_y_list.append(y_log2[chosen])
                print(f'    Borrowed {len(chosen)} from (k={nk},m={nm})')

        # Combine
        parts_X = [X_orig, extra_feats]
        parts_y = [y_orig, extra_y]
        if cross_X_list:
            parts_X.extend(cross_X_list)
            parts_y.extend(cross_y_list)

        X_all = np.vstack(parts_X)
        y_all = np.vstack(parts_y)
        X_all = np.nan_to_num(X_all, nan=0.0, posinf=100.0, neginf=-100.0)

        # Target clipping
        y_p5, y_p95 = np.percentile(y_all, 2), np.percentile(y_all, 98)
        y_clipped = np.clip(y_all, y_p5, y_p95)
        n_clipped = np.sum(y_all != y_clipped)
        if n_clipped > 0:
            print(f'    Clipped {n_clipped} targets to [{y_p5:.2f}, {y_p95:.2f}]')

        # Scaler on ALL data
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_all)

        # Split: original val pure
        n_orig = len(X_orig)
        np.random.seed(42)
        orig_perm = np.random.permutation(n_orig)
        orig_val_size = int(0.1 * n_orig)
        orig_val_idx = orig_perm[:orig_val_size]
        orig_train_idx = orig_perm[orig_val_size:]

        train_indices = list(orig_train_idx) + list(range(n_orig, len(X_scaled)))
        val_indices = list(orig_val_idx)

        train_ds = MHeightDataset(X_scaled[train_indices], y_clipped[train_indices])
        val_ds = MHeightDataset(X_scaled[val_indices], y_all[val_indices])  # unclipped val

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  drop_last=True, num_workers=NUM_WORKERS, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)

        weak_group_data[gk] = {
            'scaler': scaler, 'train_loader': train_loader, 'val_loader': val_loader,
            'train_size': len(train_indices), 'val_size': len(val_indices),
            'feature_dim': X_scaled.shape[1],
        }
        print(f'  (k={k_val},m={m_val}): {n_orig} orig + {len(extra_feats)} LP + '
              f'{sum(len(cx) for cx in cross_X_list) if cross_X_list else 0} cross '
              f'= {len(X_all)} total, train={len(train_indices)}, val={len(val_indices)}')

## Cell 16: STEP 9 — Retrain Experts for Weak Groups

Retrain a new expert for each weak group using the **augmented dataset** (original + LP + cross-group). These retrained experts use:
- Larger architectures (up to 1024 hidden, 8 blocks)
- **Combined loss** (Huber + MSE + log-cosh)
- Higher noise injection for regularization
- **Mixup augmentation** (α=0.2)
- More training epochs (1200) and patience (up to 100)
- Best-of-15 runs

| Tier | Groups | Hidden | Blocks | Patience | Noise | Runs |
|------|--------|--------|--------|----------|-------|------|
| Hard | (4,5), (5,4), (6,3) | 1024 | 8 | 100 | 0.05-0.06 | 15 |
| Medium | (4,2), (4,4), (5,3) | 1024 | 8 | 80 | 0.04-0.05 | 15 |
| Easy | (4,3), (5,2), (6,2) | 768 | 6 | 70 | 0.03 | 15 |

In [ ]:
# =========================================================================
# STEP 9: Retrain Base Experts for Weak Groups
# =========================================================================
if weak_groups:
    print('\n' + '=' * 70)
    print('STEP 9: Retrain Base Experts (Weak Groups) — Scaled Up')
    print('=' * 70)

    RETRAIN_CONFIGS = {
        # (hidden, blocks, dropout, wd, lr, epochs, patience, n_runs, noise_std)
        # Hard groups
        (4, 5): (1024, 8, 0.18, 5e-4, 2e-4, WEAK_EPOCHS, 100, RETRAIN_RUNS, 0.06),
        (5, 4): (1024, 8, 0.18, 5e-4, 2e-4, WEAK_EPOCHS, 100, RETRAIN_RUNS, 0.06),
        (6, 3): (1024, 8, 0.15, 3e-4, 2e-4, WEAK_EPOCHS, 100, RETRAIN_RUNS, 0.05),
        # Medium groups
        (4, 2): (1024, 8, 0.15, 3e-4, 2e-4, WEAK_EPOCHS, 80, RETRAIN_RUNS, 0.04),
        (4, 4): (1024, 8, 0.15, 3e-4, 2e-4, WEAK_EPOCHS, 80, RETRAIN_RUNS, 0.05),
        (5, 3): (1024, 8, 0.15, 3e-4, 2e-4, WEAK_EPOCHS, 80, RETRAIN_RUNS, 0.05),
        # Easy groups
        (4, 3): (768,  6, 0.15, 3e-4, 3e-4, WEAK_EPOCHS, 70, RETRAIN_RUNS, 0.03),
        (5, 2): (768,  6, 0.15, 3e-4, 3e-4, WEAK_EPOCHS, 70, RETRAIN_RUNS, 0.03),
        (6, 2): (768,  6, 0.12, 3e-4, 3e-4, WEAK_EPOCHS, 70, RETRAIN_RUNS, 0.03),
    }

    retrained_experts = {}
    retrained_scalers = {}

    for gk in weak_groups:
        if gk not in RETRAIN_CONFIGS:
            RETRAIN_CONFIGS[gk] = (768, 6, 0.15, 3e-4, 3e-4, WEAK_EPOCHS, 60, RETRAIN_RUNS, 0.04)
        h_dim, n_blk, drop, wd, lr, max_ep, patience, n_runs, noise_std = RETRAIN_CONFIGS[gk]
        info = weak_group_data[gk]

        print(f"\n{'='*60}")
        print(f'RETRAIN Expert (k={gk[0]},m={gk[1]}): h={h_dim}, b={n_blk}, '
              f'runs={n_runs}')
        print(f'  Data: {info["train_size"]} train, {info["val_size"]} val')
        print(f'  Old cost: {group_costs[gk]:.4f}')
        print(f"{'='*60}")

        global_best_val = float('inf')
        global_best_state = None

        for run in range(n_runs):
            print(f'\n    --- Run {run+1}/{n_runs} ---')
            seed = 1000 + run * 333 + gk[0] * 100 + gk[1] * 10
            model = ExpertDNN(info['feature_dim'], h_dim, n_blk, drop).to(device)
            val_loss, state, _, _ = train_single_run(
                model, info['train_loader'], info['val_loader'],
                info['train_size'], info['val_size'], device,
                lr=lr, weight_decay=wd, max_epochs=max_ep,
                patience=patience, noise_std=noise_std,
                use_combined_loss=True, seed=seed,
                use_mixup=True, mixup_alpha=0.2,
            )
            print(f'    Run {run+1} best val: {val_loss:.6f}')
            if val_loss < global_best_val:
                global_best_val = val_loss
                global_best_state = state

        model = ExpertDNN(info['feature_dim'], h_dim, n_blk, drop).to(device)
        model.load_state_dict(global_best_state)
        model.eval()
        retrained_experts[gk] = model
        retrained_scalers[gk] = info['scaler']
        print(f'  ✓ Best retrained val: {global_best_val:.6f}')
else:
    retrained_experts = {}
    retrained_scalers = {}

## Cell 17: STEP 10 — Diverse Ensemble per Weak Group

Build a diverse ensemble of **15 models** per weak group, each with a different architecture sampled from a pool of 20 configurations. The pool includes:
- Hidden dimensions: 768, 1024, 1280, 1536, 2048
- Block counts: 4–10
- Dropout: 0.12–0.20
- Learning rates: 1e-4, 2e-4
- Noise levels: 0.04–0.07

All ensemble models use combined loss, Mixup augmentation, and are trained for up to 1200 epochs with patience 80. The diversity in architectures helps reduce ensemble correlation and improves trimmed/weighted averaging.

In [ ]:
# =========================================================================
# STEP 10: Diverse Ensemble for Weak Groups (SCALED UP)
# =========================================================================
if weak_groups:
    print('\n' + '=' * 70)
    print(f'STEP 10: Diverse Ensemble ({ENSEMBLE_SIZE} models per weak group)')
    print('=' * 70)

    # Generate diverse architectures
    ENSEMBLE_CONFIGS = []
    arch_pool = [
        # (hidden_dim, num_blocks, dropout, lr, noise_std)
        (768,  6,  0.18, 2e-4, 0.06),
        (1024, 5,  0.18, 2e-4, 0.07),
        (768,  8,  0.15, 1e-4, 0.06),
        (1024, 6,  0.15, 2e-4, 0.06),
        (1280, 5,  0.18, 1e-4, 0.05),
        (768,  10, 0.12, 1e-4, 0.06),
        (1024, 8,  0.15, 1e-4, 0.05),
        (1536, 5,  0.18, 1e-4, 0.05),
        (1024, 10, 0.12, 1e-4, 0.05),
        (2048, 4,  0.20, 1e-4, 0.06),
        (768,  7,  0.18, 2e-4, 0.07),
        (1280, 6,  0.15, 1e-4, 0.05),
        (1024, 7,  0.15, 2e-4, 0.06),
        (1536, 6,  0.15, 1e-4, 0.05),
        (2048, 5,  0.18, 1e-4, 0.05),
        (768,  9,  0.12, 1e-4, 0.06),
        (1280, 8,  0.12, 1e-4, 0.05),
        (1536, 4,  0.20, 2e-4, 0.06),
        (2048, 6,  0.15, 1e-4, 0.04),
        (1024, 9,  0.12, 1e-4, 0.05),
    ]
    for i in range(ENSEMBLE_SIZE):
        cfg = arch_pool[i % len(arch_pool)]
        seed_offset = (i + 1) * 100
        ENSEMBLE_CONFIGS.append((*cfg, seed_offset))

    weak_ensembles = {}
    weak_scalers = {}
    ensemble_val_costs = {}

    for gk in weak_groups:
        info = weak_group_data[gk]
        weak_scalers[gk] = info['scaler']
        ens_models = []
        ens_costs = []

        print(f"\n{'='*60}")
        print(f'ENSEMBLE (k={gk[0]},m={gk[1]}): {ENSEMBLE_SIZE} models')
        print(f"{'='*60}")

        for eidx, (h_dim, n_blk, drop, lr, noise_std, seed_off) in enumerate(ENSEMBLE_CONFIGS):
            print(f'\n  Ens {eidx+1}/{ENSEMBLE_SIZE} (h={h_dim}, b={n_blk}, d={drop})')
            seed = seed_off + gk[0] * 100 + gk[1] * 10

            model = ExpertDNN(info['feature_dim'], h_dim, n_blk, drop).to(device)
            val_loss, state, _, _ = train_single_run(
                model, info['train_loader'], info['val_loader'],
                info['train_size'], info['val_size'], device,
                lr=lr, weight_decay=5e-4,
                max_epochs=ENSEMBLE_EPOCHS,
                patience=80, noise_std=noise_std,
                use_combined_loss=True, seed=seed,
                use_mixup=True, mixup_alpha=0.2,
            )
            model.load_state_dict(state)
            model.to(device)
            model.eval()
            ens_models.append(model)
            ens_costs.append(val_loss)
            print(f'    ✓ Best: {val_loss:.6f}')

        weak_ensembles[gk] = ens_models
        ensemble_val_costs[gk] = ens_costs
else:
    weak_ensembles = {}
    weak_scalers = {}
    ensemble_val_costs = {}

## Cell 18: STEP 11 — Evaluate All Strategies & Pick Best per Group

For each weak group, evaluate multiple ensemble aggregation strategies on the **original validation set** (fair comparison):

| Strategy | Description |
|----------|-------------|
| `original` | Base expert only |
| `retrained` | Retrained expert only |
| `simple` | Mean of retrained + ensemble (no original) |
| `full` | Mean of all models including original |
| `median` | Median of all models |
| `weighted` | Inverse-cost weighted average |
| `topk` | Top-5 models by inverse-cost weighting |
| `trimmed_N` | Remove worst N models, then mean |

The best strategy is selected per group and stored for use during inference. This outputs the **testing performance** (validation cost) for every strategy.

This also computes the **overall evaluation** comparing base vs. improved costs across all groups.

In [ ]:
# =========================================================================
# STEP 11: Evaluate All Strategies
# =========================================================================
if weak_groups:
    print('\n' + '=' * 70)
    print('STEP 11: Evaluate All Strategies (Trimmed, Weighted, TopK, ...)')
    print('=' * 70)

    improved_costs = {}
    ensemble_weights = {}
    best_strategy = {}

    for gk in weak_groups:
        k_val, m_val = gk

        # Get raw val features from ORIGINAL split
        orig_indices = [i for i, s in enumerate(X_combined) if s[1] == k_val and s[2] == m_val]
        X_grp = X_array[orig_indices]
        y_grp = y_log2[orig_indices]
        np.random.seed(42)
        perm = np.random.permutation(len(X_grp))
        split = int(0.9 * len(perm))
        val_idx = perm[split:]
        X_val_raw = X_grp[val_idx]
        y_val = y_grp[val_idx].flatten()

        # A) Original expert
        orig_scaler = group_data[gk]['scaler']
        orig_model = experts[gk]; orig_model.eval()
        t_orig = torch.tensor(orig_scaler.transform(X_val_raw), dtype=torch.float32).to(device)
        orig_preds = batch_predict(orig_model, t_orig)
        orig_cost = ((y_val - orig_preds) ** 2).mean()

        # B) Retrained expert
        ret_preds = None
        ret_cost = float('inf')
        if gk in retrained_experts:
            ret_scaler = retrained_scalers[gk]
            ret_model = retrained_experts[gk]; ret_model.eval()
            t_ret = torch.tensor(ret_scaler.transform(X_val_raw), dtype=torch.float32).to(device)
            ret_preds = batch_predict(ret_model, t_ret)
            ret_cost = ((y_val - ret_preds) ** 2).mean()

        # C) Ensemble
        ens_pred_arrays = []
        if gk in weak_ensembles:
            ens_scaler = weak_scalers[gk]
            t_ens = torch.tensor(ens_scaler.transform(X_val_raw), dtype=torch.float32).to(device)
            for em in weak_ensembles[gk]:
                em.eval()
                ens_pred_arrays.append(batch_predict(em, t_ens))

        # Build pools
        pool_no_orig = []
        if ret_preds is not None:
            pool_no_orig.append(ret_preds)
        pool_no_orig.extend(ens_pred_arrays)
        pool_with_orig = [orig_preds] + pool_no_orig

        results = {'original': orig_cost}
        if ret_preds is not None:
            results['retrained'] = ret_cost

        if pool_no_orig:
            arr_no = np.array(pool_no_orig)
            arr_full = np.array(pool_with_orig)

            results['simple'] = ((y_val - np.mean(arr_no, axis=0)) ** 2).mean()
            results['full'] = ((y_val - np.mean(arr_full, axis=0)) ** 2).mean()
            results['median'] = ((y_val - np.median(arr_full, axis=0)) ** 2).mean()

            pool_costs = [orig_cost]
            if ret_preds is not None:
                pool_costs.append(ret_cost)
            pool_costs.extend(ensemble_val_costs.get(gk, [1.0] * len(ens_pred_arrays)))
            inv = [1.0 / (c + 1e-8) for c in pool_costs]
            w = np.array([x / sum(inv) for x in inv])
            weighted_avg = np.average(arr_full, axis=0, weights=w)
            results['weighted'] = ((y_val - weighted_avg) ** 2).mean()

            K = min(5, len(pool_costs))
            si = np.argsort(pool_costs)[:K]
            tw = w[si]; tw = tw / tw.sum()
            topk_avg = np.average(arr_full[si], axis=0, weights=tw)
            results['topk'] = ((y_val - topk_avg) ** 2).mean()

            for trim_n in [2, 3, 4]:
                if len(arr_full) > trim_n + 3:
                    per_model = [((y_val - arr_full[i]) ** 2).mean() for i in range(len(arr_full))]
                    keep = np.argsort(per_model)[:len(arr_full) - trim_n]
                    trimmed = np.mean(arr_full[keep], axis=0)
                    tag = f'trimmed_{trim_n}'
                    results[tag] = ((y_val - trimmed) ** 2).mean()

        best_name = min(results, key=results.get)
        best_val_cost = results[best_name]

        print(f'\n  (k={k_val},m={m_val}): orig={orig_cost:.4f} → best={best_name}({best_val_cost:.4f})')
        for name, val in sorted(results.items(), key=lambda x: x[1]):
            print(f'    {name:>15}: {val:.4f}')

        # Store trimmed_keep for the winning strategy
        trimmed_keep = None
        if best_name.startswith('trimmed_'):
            trim_n = int(best_name.split('_')[1])
            per_model_final = [((y_val - np.array(pool_with_orig)[i]) ** 2).mean()
                               for i in range(len(pool_with_orig))]
            trimmed_keep = np.argsort(per_model_final)[:len(pool_with_orig) - trim_n].tolist()
        elif best_name == 'simple':
            trimmed_keep = None
        elif best_name == 'full':
            trimmed_keep = list(range(len(pool_with_orig)))

        improved_costs[gk] = {
            'old_cost': float(orig_cost),
            'new_cost': float(best_val_cost),
            'strategy': best_name,
            'weights': w.tolist() if 'weighted' in results else None,
            'topk_indices': si.tolist() if 'topk' in results else None,
            'topk_weights': tw.tolist() if 'topk' in results else None,
            'trimmed_keep': trimmed_keep,
        }
        best_strategy[gk] = best_name
        ensemble_weights[gk] = w.tolist() if 'weighted' in results else []

    # Determine which groups improved
    use_ensemble_for = []
    for gk in weak_groups:
        if improved_costs[gk]['new_cost'] < improved_costs[gk]['old_cost']:
            use_ensemble_for.append(gk)

    # Overall evaluation
    print('\n' + '-' * 60)
    print('OVERALL EVALUATION:')
    total_cost_sum_new, total_count_new = 0.0, 0
    for gk in group_keys:
        if gk in improved_costs and improved_costs[gk]['new_cost'] < improved_costs[gk]['old_cost']:
            cost = improved_costs[gk]['new_cost']
            k_val, m_val = gk
            orig_idx = [i for i, s in enumerate(X_combined) if s[1] == k_val and s[2] == m_val]
            n_samp = len(orig_idx) - int(0.9 * len(orig_idx))
            total_cost_sum_new += cost * n_samp
            total_count_new += n_samp
            source = improved_costs[gk]['strategy']
        else:
            model = experts[gk]; model.eval()
            preds_l, trues_l = [], []
            with torch.no_grad():
                for bx, by in group_data[gk]['val_loader']:
                    bx = bx.to(device)
                    preds_l.append(model(bx).cpu().numpy())
                    trues_l.append(by.numpy())
            pa = np.concatenate(preds_l).flatten()
            ta = np.concatenate(trues_l).flatten()
            cost = ((ta - pa) ** 2).mean()
            total_cost_sum_new += ((ta - pa) ** 2).sum()
            total_count_new += len(ta)
            source = 'original'
            n_samp = len(ta)
        print(f'  (k={gk[0]},m={gk[1]}): cost={cost:.6f} [{source}] ({n_samp} samples)')

    new_overall = total_cost_sum_new / total_count_new
    print(f'\n  Base overall:  {overall_base:.6f}')
    print(f'  New overall:   {new_overall:.6f}')
    print(f'  Improvement:   {overall_base - new_overall:+.6f}')
else:
    use_ensemble_for = []
    best_strategy = {}
    ensemble_weights = {}
    improved_costs = {}

## Cell 19: STEP 12 — Save Model

Save the complete model to `moe_model.pth` including:
- All 9 base expert state dicts + architecture configs + scalers
- Retrained expert state dicts + scalers for improved groups
- Ensemble model state dicts + scalers for improved groups
- Best strategy, weights, and trimmed indices per group

This is the file to submit to Canvas.

In [ ]:
# =========================================================================
# STEP 12: Save Model
# =========================================================================
print('\n' + '=' * 70)
print('STEP 12: Save Model')
print('=' * 70)

save_dict = {
    'feature_dim': FEATURE_DIM,
    'group_keys': group_keys,
    'ensemble_groups': use_ensemble_for if weak_groups else [],
    'ensemble_size': ENSEMBLE_SIZE if weak_groups else 0,
}

# Base experts
for gk in group_keys:
    ks = f'k{gk[0]}_m{gk[1]}'
    save_dict[f'expert_{ks}_state'] = experts[gk].state_dict()
    save_dict[f'scaler_{ks}'] = group_data[gk]['scaler']
    h, b, d = expert_hparams[gk]
    save_dict[f'expert_{ks}_hidden_dim'] = h
    save_dict[f'expert_{ks}_num_blocks'] = b
    save_dict[f'expert_{ks}_dropout'] = d

# Retrained + ensemble for improved groups
if weak_groups:
    for gk in use_ensemble_for:
        ks = f'k{gk[0]}_m{gk[1]}'
        strategy = best_strategy.get(gk, 'simple')
        save_dict[f'ensemble_{ks}_strategy'] = strategy

        if gk in retrained_experts:
            save_dict[f'retrained_{ks}_state'] = retrained_experts[gk].state_dict()
            save_dict[f'retrained_scaler_{ks}'] = retrained_scalers[gk]
            cfg = RETRAIN_CONFIGS[gk]
            save_dict[f'retrained_{ks}_hidden_dim'] = cfg[0]
            save_dict[f'retrained_{ks}_num_blocks'] = cfg[1]
            save_dict[f'retrained_{ks}_dropout'] = cfg[2]

        if gk in weak_ensembles:
            save_dict[f'ensemble_scaler_{ks}'] = weak_scalers[gk]
            for eidx, em in enumerate(weak_ensembles[gk]):
                save_dict[f'ensemble_{ks}_model{eidx}_state'] = em.state_dict()
                ea = ENSEMBLE_CONFIGS[eidx]
                save_dict[f'ensemble_{ks}_model{eidx}_hidden_dim'] = ea[0]
                save_dict[f'ensemble_{ks}_model{eidx}_num_blocks'] = ea[1]
                save_dict[f'ensemble_{ks}_model{eidx}_dropout'] = ea[2]

        if gk in ensemble_weights:
            save_dict[f'ensemble_{ks}_weights'] = ensemble_weights[gk]
        if gk in improved_costs:
            ic = improved_costs[gk]
            if ic.get('topk_indices'):
                save_dict[f'ensemble_{ks}_topk_indices'] = ic['topk_indices']
                save_dict[f'ensemble_{ks}_topk_weights'] = ic['topk_weights']
            if ic.get('trimmed_keep'):
                save_dict[f'ensemble_{ks}_trimmed_keep'] = ic['trimmed_keep']

model_path = os.path.join(OUTPUT_DIR, 'moe_model.pth')
torch.save(save_dict, model_path)
file_size = os.path.getsize(model_path) / (1024 * 1024)
print(f"Model saved to '{model_path}' ({file_size:.1f} MB)")
print(f'  {len(group_keys)} base experts')
if weak_groups and use_ensemble_for:
    print(f'  {len(use_ensemble_for)} ensemble groups × {ENSEMBLE_SIZE} models')
    for gk in use_ensemble_for:
        print(f"    (k={gk[0]},m={gk[1]}): {best_strategy.get(gk, 'simple')}")

## Cell 20: STEP 13 — Generate Predictions (Self-Evaluation on Training Data)

Generate predictions on the training data itself as a self-evaluation sanity check. For each sample:
1. Route to the correct $(k,m)$ group expert
2. If the group has an improved ensemble, use the best strategy (trimmed, weighted, etc.)
3. Convert predictions from $\log_2$-space back to linear space: $\hat{y} = \max(2^{\hat{y}_{\log_2}}, 1.0)$

The output is a list of predicted m-heights in the exact same format as `CSCE-636-Project-1-Train-mHeights`.

In [ ]:
# =========================================================================
# STEP 13: Generate Predictions (test = training data for self-eval)
# =========================================================================
print('\n' + '=' * 70)
print('STEP 13: Generate Predictions')
print('=' * 70)

test_path = os.path.join(DATA_DIR, 'CSCE-636-Project-1-Train-n_k_m_P')
with open(test_path, 'rb') as f:
    test_data = pickle.load(f)

test_groups = defaultdict(list)
for i, sample in enumerate(test_data):
    n, k, m, P = sample
    test_groups[(k, m)].append((i, extract_features(sample)))

predicted_mheights = [0.0] * len(test_data)

for gk, items in test_groups.items():
    indices = [it[0] for it in items]
    features = np.array([it[1] for it in items], dtype=np.float32)

    if gk not in experts:
        for idx in indices:
            predicted_mheights[idx] = 1.0
        continue

    # Original expert
    orig_sc = group_data[gk]['scaler'].transform(features)
    orig_t = torch.tensor(orig_sc, dtype=torch.float32).to(device)
    orig_preds = batch_predict(experts[gk], orig_t)

    if gk in (use_ensemble_for if weak_groups else []) and gk in weak_ensembles:
        strategy = best_strategy.get(gk, 'simple')

        all_pred_arrays = []
        if gk in retrained_experts:
            ret_sc = retrained_scalers[gk].transform(features)
            ret_t = torch.tensor(ret_sc, dtype=torch.float32).to(device)
            all_pred_arrays.append(batch_predict(retrained_experts[gk], ret_t))

        ens_sc = weak_scalers[gk].transform(features)
        ens_t = torch.tensor(ens_sc, dtype=torch.float32).to(device)
        for em in weak_ensembles[gk]:
            all_pred_arrays.append(batch_predict(em, ens_t))

        arr = np.array(all_pred_arrays)

        if strategy == 'weighted' and gk in ensemble_weights:
            w = np.array(ensemble_weights[gk])
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.average(full_arr, axis=0, weights=w[:len(full_arr)])
        elif strategy == 'topk' and gk in improved_costs and improved_costs[gk].get('topk_indices'):
            tk_idx = improved_costs[gk]['topk_indices']
            tk_w = np.array(improved_costs[gk]['topk_weights'])
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            ti = [i for i in tk_idx if i < len(full_arr)]
            tw = tk_w[:len(ti)]
            tw = tw / tw.sum()
            preds_log2 = np.average(full_arr[ti], axis=0, weights=tw)
        elif strategy.startswith('trimmed') and gk in improved_costs and improved_costs[gk].get('trimmed_keep'):
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            keep_idx = [i for i in improved_costs[gk]['trimmed_keep'] if i < len(full_arr)]
            preds_log2 = np.mean(full_arr[keep_idx], axis=0)
        elif strategy == 'median':
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.median(full_arr, axis=0)
        elif strategy == 'full':
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.mean(full_arr, axis=0)
        else:
            preds_log2 = np.mean(arr, axis=0)

        source = f'ens-{strategy}({len(all_pred_arrays)})'
    else:
        preds_log2 = orig_preds
        source = 'single'

    preds = np.maximum(2.0 ** preds_log2, 1.0)
    for idx, val in zip(indices, preds):
        predicted_mheights[idx] = float(val)

    print(f'  (k={gk[0]},m={gk[1]}) [{source:>25}]: {len(indices)} samples, '
          f'range=[{preds.min():.2f}, {preds.max():.2f}]')

pred_path = os.path.join(OUTPUT_DIR, 'predicted_mHeights')
with open(pred_path, 'wb') as f:
    pickle.dump(predicted_mheights, f)

print(f"\nPredictions saved to '{pred_path}'")
print(f'Total: {len(predicted_mheights)}')
print(f'Range: [{min(predicted_mheights):.4f}, {max(predicted_mheights):.4f}]')
print(f'All >= 1: {all(h >= 1.0 for h in predicted_mheights)}')

print('\n' + '=' * 70)
print('TRAINING DONE! 🎉')
print('=' * 70)

---

## Cell 21: Final Inference Cell — For Grader Testing

**This is the standalone inference block for the grader.** It does the following:

1. **Loads the saved pre-trained DNN model** (`moe_model.pth`)
2. **Accepts a test dataset** where each sample is `[n, k, m, P]` (loaded from a pickle file)
3. **Outputs a list of predicted m-heights** in the exact same format as `CSCE-636-Project-1-Train-mHeights`

**Usage:** Set `TEST_DATA_PATH` to point to the test file (e.g., `CSCE-636-Project-1-Train-n_k_m_P` or any future Sample Test Set), and `MODEL_PATH` to the saved model. Run this cell to generate predictions.

The predictions are saved via `pickle.dump()` as required for submission.

In [ ]:
# =============================================================================
# FINAL INFERENCE CELL — Standalone block for grader testing
# =============================================================================
# Set these paths before running:
MODEL_PATH = 'moe_model.pth'                           # Path to saved model
TEST_DATA_PATH = 'CSCE-636-Project-1-Train-n_k_m_P'    # Path to test data file
OUTPUT_PRED_PATH = 'predicted_mHeights'                 # Path to save predictions

import pickle
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict

# ---- Architecture definitions (must match training) ----
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.block(x) + x)

class ExpertDNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=3, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
        )
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)]
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1),
        )
    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_blocks(x)
        return self.output_head(x)

# ---- Feature extraction (must match training) ----
def extract_features(sample):
    n, k, m, P = sample
    max_k, max_nk, max_sv = 6, 5, 5
    P_padded = np.zeros((max_k, max_nk))
    P_padded[:k, :n-k] = P
    P_flat = P_padded.flatten()
    row_norms = np.zeros(max_k)
    for i in range(k): row_norms[i] = np.linalg.norm(P[i, :])
    col_norms = np.zeros(max_nk)
    for j in range(n - k): col_norms[j] = np.linalg.norm(P[:, j])
    sv = np.zeros(max_sv)
    try:
        svs = np.linalg.svd(P, compute_uv=False)
        sv[:len(svs)] = np.sort(svs)[::-1]
    except: pass
    frob_norm = np.linalg.norm(P, 'fro')
    max_abs = np.max(np.abs(P))
    mean_abs = np.mean(np.abs(P))
    std_abs = np.std(np.abs(P))
    min_abs = np.min(np.abs(P))
    try: cond = np.log1p(np.linalg.cond(P))
    except: cond = 0.0
    if not np.isfinite(cond): cond = 50.0
    mean_row = np.mean(row_norms[:k]) if k > 0 else 1.0
    mean_col = np.mean(col_norms[:n-k]) if (n - k) > 0 else 1.0
    ratio = mean_row / (mean_col + 1e-8)
    G = np.hstack([np.eye(k), P])
    g_col_norms = np.zeros(9)
    for j in range(n): g_col_norms[j] = np.linalg.norm(G[:, j])
    g_col_max = np.max(g_col_norms)
    g_col_min = np.min(g_col_norms[:n])
    g_col_mean = np.mean(g_col_norms[:n])
    g_col_std = np.std(g_col_norms[:n])
    try: rank = np.linalg.matrix_rank(P)
    except: rank = min(k, n - k)
    effective_rank = np.sum(sv > 1e-6)
    sv_ratio = sv[0] / (sv[min(k, n-k)-1] + 1e-10) if sv[min(k, n-k)-1] > 1e-10 else 100.0
    sv_sum = np.sum(sv)
    sv_energy_ratio = sv[0] / (sv_sum + 1e-10)
    try:
        PtP = P.T @ P
        ptP_trace = np.trace(PtP)
        ptP_frob = np.linalg.norm(PtP, 'fro')
    except:
        ptP_trace = frob_norm ** 2
        ptP_frob = frob_norm ** 2
    try:
        sq = min(k, n - k)
        det_val = np.log1p(abs(np.linalg.det(P[:sq, :sq])))
    except: det_val = 0.0
    if not np.isfinite(det_val): det_val = 0.0
    return np.concatenate([
        [n, k, m], P_flat, row_norms, col_norms, sv,
        [frob_norm, max_abs, mean_abs, std_abs, min_abs],
        [cond, ratio],
        g_col_norms, [g_col_max, g_col_min, g_col_mean, g_col_std],
        [rank, effective_rank],
        [sv_ratio, sv_sum, sv_energy_ratio],
        [ptP_trace, ptP_frob],
        [det_val],
    ])

def batch_predict(model, tensor, batch_size=2048):
    preds = []
    model.eval()
    with torch.no_grad():
        for bi in range(0, len(tensor), batch_size):
            preds.append(model(tensor[bi:bi+batch_size]).cpu().numpy().flatten())
    return np.concatenate(preds)

# ---- Load model ----
print('Loading model...')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)

group_keys = checkpoint['group_keys']
ensemble_groups = checkpoint.get('ensemble_groups', [])
ensemble_size = checkpoint.get('ensemble_size', 0)
feature_dim = checkpoint.get('feature_dim', 77)

# Rebuild base experts
experts = {}
scalers = {}
for gk in group_keys:
    ks = f'k{gk[0]}_m{gk[1]}'
    h = checkpoint[f'expert_{ks}_hidden_dim']
    b = checkpoint[f'expert_{ks}_num_blocks']
    d = checkpoint[f'expert_{ks}_dropout']
    model = ExpertDNN(feature_dim, h, b, d).to(device)
    model.load_state_dict(checkpoint[f'expert_{ks}_state'])
    model.eval()
    experts[gk] = model
    scalers[gk] = checkpoint[f'scaler_{ks}']

# Rebuild retrained + ensemble for improved groups
retrained_experts = {}
retrained_scalers = {}
weak_ensembles = {}
weak_scalers = {}
strategies = {}
ens_weights = {}
ens_topk = {}
ens_trimmed = {}

for gk in ensemble_groups:
    ks = f'k{gk[0]}_m{gk[1]}'
    strategies[gk] = checkpoint.get(f'ensemble_{ks}_strategy', 'simple')

    # Retrained expert
    ret_key = f'retrained_{ks}_state'
    if ret_key in checkpoint:
        h = checkpoint[f'retrained_{ks}_hidden_dim']
        b = checkpoint[f'retrained_{ks}_num_blocks']
        d = checkpoint[f'retrained_{ks}_dropout']
        m = ExpertDNN(feature_dim, h, b, d).to(device)
        m.load_state_dict(checkpoint[ret_key])
        m.eval()
        retrained_experts[gk] = m
        retrained_scalers[gk] = checkpoint.get(f'retrained_scaler_{ks}')

    # Ensemble models
    ens_scaler_key = f'ensemble_scaler_{ks}'
    if ens_scaler_key in checkpoint:
        weak_scalers[gk] = checkpoint[ens_scaler_key]
        ens_list = []
        for eidx in range(ensemble_size):
            state_key = f'ensemble_{ks}_model{eidx}_state'
            if state_key not in checkpoint:
                break
            h = checkpoint[f'ensemble_{ks}_model{eidx}_hidden_dim']
            b = checkpoint[f'ensemble_{ks}_model{eidx}_num_blocks']
            d = checkpoint[f'ensemble_{ks}_model{eidx}_dropout']
            m = ExpertDNN(feature_dim, h, b, d).to(device)
            m.load_state_dict(checkpoint[state_key])
            m.eval()
            ens_list.append(m)
        weak_ensembles[gk] = ens_list

    # Weights / topk / trimmed
    w_key = f'ensemble_{ks}_weights'
    if w_key in checkpoint:
        ens_weights[gk] = checkpoint[w_key]
    tk_key = f'ensemble_{ks}_topk_indices'
    if tk_key in checkpoint:
        ens_topk[gk] = {
            'indices': checkpoint[tk_key],
            'weights': checkpoint.get(f'ensemble_{ks}_topk_weights', []),
        }
    tr_key = f'ensemble_{ks}_trimmed_keep'
    if tr_key in checkpoint:
        ens_trimmed[gk] = checkpoint[tr_key]

print(f'Loaded {len(experts)} base experts, {len(ensemble_groups)} ensemble groups')

# ---- Load test data ----
print(f'Loading test data from: {TEST_DATA_PATH}')
with open(TEST_DATA_PATH, 'rb') as f:
    test_data = pickle.load(f)
print(f'Test samples: {len(test_data)}')

# ---- Generate predictions ----
test_groups = defaultdict(list)
for i, sample in enumerate(test_data):
    n, k, m, P = sample
    test_groups[(k, m)].append((i, extract_features(sample)))

predicted_mheights = [0.0] * len(test_data)

for gk, items in test_groups.items():
    indices = [it[0] for it in items]
    features = np.array([it[1] for it in items], dtype=np.float32)

    if gk not in experts:
        for idx in indices:
            predicted_mheights[idx] = 1.0
        continue

    # Base expert prediction
    orig_sc = scalers[gk].transform(features)
    orig_t = torch.tensor(orig_sc, dtype=torch.float32).to(device)
    orig_preds = batch_predict(experts[gk], orig_t)

    if gk in ensemble_groups and gk in weak_ensembles:
        strategy = strategies.get(gk, 'simple')

        all_pred_arrays = []
        if gk in retrained_experts:
            ret_sc = retrained_scalers[gk].transform(features)
            ret_t = torch.tensor(ret_sc, dtype=torch.float32).to(device)
            all_pred_arrays.append(batch_predict(retrained_experts[gk], ret_t))

        ens_sc = weak_scalers[gk].transform(features)
        ens_t = torch.tensor(ens_sc, dtype=torch.float32).to(device)
        for em in weak_ensembles[gk]:
            all_pred_arrays.append(batch_predict(em, ens_t))

        arr = np.array(all_pred_arrays)

        if strategy == 'weighted' and gk in ens_weights:
            w = np.array(ens_weights[gk])
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.average(full_arr, axis=0, weights=w[:len(full_arr)])
        elif strategy == 'topk' and gk in ens_topk:
            tk_idx = ens_topk[gk]['indices']
            tk_w = np.array(ens_topk[gk]['weights'])
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            ti = [i for i in tk_idx if i < len(full_arr)]
            tw = tk_w[:len(ti)]; tw = tw / tw.sum()
            preds_log2 = np.average(full_arr[ti], axis=0, weights=tw)
        elif strategy.startswith('trimmed') and gk in ens_trimmed:
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            keep_idx = [i for i in ens_trimmed[gk] if i < len(full_arr)]
            preds_log2 = np.mean(full_arr[keep_idx], axis=0)
        elif strategy == 'median':
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.median(full_arr, axis=0)
        elif strategy == 'full':
            full_arr = np.vstack([orig_preds[np.newaxis, :], arr])
            preds_log2 = np.mean(full_arr, axis=0)
        else:
            preds_log2 = np.mean(arr, axis=0)

        source = f'ens-{strategy}({len(all_pred_arrays)})'
    else:
        preds_log2 = orig_preds
        source = 'single'

    preds = np.maximum(2.0 ** preds_log2, 1.0)
    for idx, val in zip(indices, preds):
        predicted_mheights[idx] = float(val)

    print(f'  (k={gk[0]},m={gk[1]}) [{source:>25}]: {len(indices)} samples, '
          f'range=[{preds.min():.2f}, {preds.max():.2f}]')

# ---- Save predictions ----
with open(OUTPUT_PRED_PATH, 'wb') as f:
    pickle.dump(predicted_mheights, f)

print(f'\n✅ Predictions saved to: {OUTPUT_PRED_PATH}')
print(f'   Total predictions: {len(predicted_mheights)}')
print(f'   Range: [{min(predicted_mheights):.4f}, {max(predicted_mheights):.4f}]')
print(f'   All >= 1.0: {all(h >= 1.0 for h in predicted_mheights)}')